## Imports

In [ ]:
# Pick up edits to dataset_helpers / models without restarting the kernel.

from dotenv import load_dotenv
from datasets import load_dataset

from dataset_helpers import rank_by_price, report_top_prices
from notebooks.item_loader import DATASET, ChunkedItemLoader

load_dotenv(override=True)

## Load our Datasets

In [ ]:
# Needs the repo's loading script, so datasets is pinned to 3.6.0 (4.0 dropped script support).
# Note: streaming=True fails here — pyarrow can't do the cast the script's schema implies.

CATEGORY = "Appliances"

dataset = load_dataset(
    DATASET,
    f"raw_meta_{CATEGORY}",
    split="full",
    trust_remote_code=True,
)

## Find the most Expensive one.

In [ ]:
ranked = rank_by_price(dataset)
report_top_prices(dataset, ranked)

## Construct Items objects

In [ ]:
# Hand the loader the dataset we already have: the price report above needs the
# full 16 columns, while the loader prunes its own copy down to the five that
# Item reads and decodes the rows across a process pool.
loader = ChunkedItemLoader(CATEGORY, source=dataset)
items = loader.load()

print(f"{len(items):,} of {len(dataset):,} datapoints became items")
print(f"{sum(item.weight is not None for item in items):,} of those carry a weight")
items[1]